# RoadSense: train the road damage detector (M2, M3, M4)

Runs on a **Kaggle GPU notebook** (Settings → Accelerator → GPU T4 x2 or P100, Internet on).

1. Add the RDD2022 dataset as an input (*Add data* → search "RDD2022"), or set `RDD_ROOT` below.
2. Add a Kaggle secret `HF_TOKEN` (a Hugging Face write token) for the upload step.
3. Run all. Checkpoints go to `/kaggle/working`, so a session that times out can resume.

Outputs: `results/baseline/*` (metrics + plots for the report) and ONNX models on the Hugging Face Hub.

In [ ]:
%pip install -q "ultralytics>=8.3" onnx onnxslim huggingface_hub gpxpy
!git clone -q --depth 1 https://github.com/NinadPandith/Road-Sense.git /kaggle/working/Road-Sense || true
%cd /kaggle/working/Road-Sense/ml

In [ ]:
import glob, json, os, shutil
from pathlib import Path

# Find the RDD2022 folder in the Kaggle inputs (it contains India/, Japan/, ...).
candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/India/train", recursive=True)]
RDD_ROOT = candidates[0].parent if candidates else Path("/kaggle/input/rdd2022/RDD2022")
print(
    "RDD_ROOT =", RDD_ROOT, "| countries:", sorted(p.name for p in RDD_ROOT.iterdir() if p.is_dir())
)

WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
HF_REPO = "YOUR-HF-USER/roadsense-models"  # <- change me
VERSION = "v1-rdd2022"

## M2: convert to YOLO format (India first, then all countries)

In [ ]:
!python -m scripts.prepare_rdd2022 --src "$RDD_ROOT" --out /kaggle/working/data/rdd_india --countries India
!python -m scripts.prepare_rdd2022 --src "$RDD_ROOT" --out /kaggle/working/data/rdd_all
for name in ("rdd_india", "rdd_all"):
    stats = json.loads((WORK / "data" / name / "stats.json").read_text())
    print(name, stats["images"], stats["class_counts"])

## M3: train the baseline YOLO small

India-only first (fast sanity check), then all countries. 100 epochs of YOLO small on all of RDD2022
fits in one ~12 h Kaggle session; reduce `EPOCHS` if your quota is low. `resume=True` continues
from `last.pt` after a timeout.

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
MODEL = "yolo11s.pt"  # newest Ultralytics release at the time; change if a newer one exists


def train(data, name, model=MODEL, epochs=EPOCHS, batch=32):
    last = WORK / "runs" / name / "weights" / "last.pt"
    if last.exists():
        return YOLO(str(last)).train(resume=True)
    return YOLO(model).train(
        data=str(data),
        epochs=epochs,
        imgsz=640,
        batch=batch,
        project=str(WORK / "runs"),
        name=name,
        exist_ok=True,
        patience=25,
        seed=42,
        plots=True,
    )


train(WORK / "data/rdd_india/data.yaml", "small_india", epochs=50)
train(WORK / "data/rdd_all/data.yaml", "small_all")

## Evaluate on the held-out test split (India subset reported separately)

In [ ]:
def evaluate(weights, data, tag):
    m = YOLO(str(weights)).val(
        data=str(data),
        split="test",
        imgsz=640,
        plots=True,
        project=str(WORK / "eval"),
        name=tag,
        exist_ok=True,
    )
    names = m.names
    out = {
        "map50": float(m.box.map50),
        "map50_95": float(m.box.map),
        "per_class": {
            names[i]: {
                "precision": float(m.box.p[k]),
                "recall": float(m.box.r[k]),
                "map50": float(m.box.ap50[k]),
            }
            for k, i in enumerate(m.box.ap_class_index)
        },
        "speed_ms": m.speed,
    }
    d = RESULTS / "baseline" / tag
    d.mkdir(parents=True, exist_ok=True)
    (d / "metrics.json").write_text(json.dumps(out, indent=2))
    for f in (WORK / "eval" / tag).glob("*.png"):
        shutil.copy2(f, d / f.name)
    print(tag, json.dumps(out, indent=2))
    return out


best_small = WORK / "runs/small_all/weights/best.pt"
evaluate(best_small, WORK / "data/rdd_all/data.yaml", "small_all_test")
evaluate(best_small, WORK / "data/rdd_india/data.yaml", "small_india_test")

## Train the nano model for the browser (Drive mode)

In [ ]:
train(WORK / "data/rdd_all/data.yaml", "nano_all", model="yolo11n.pt", batch=64)
evaluate(WORK / "runs/nano_all/weights/best.pt", WORK / "data/rdd_all/data.yaml", "nano_all_test")

## M4: export to ONNX and upload to the Hugging Face Hub

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python -m scripts.export_and_upload --small {best_small} --nano /kaggle/working/runs/nano_all/weights/best.pt --repo {HF_REPO} --version {VERSION} --out /kaggle/working/export

## Next

* Download `/kaggle/working/results` and commit it to `ml/results/` (it feeds the report and the results page).
* Set `HF_MODEL_REPO` and `MODEL_VERSION` in the backend environment.
* Copy `yolo-nano.onnx` to `frontend/public/models/` for Drive mode (it is git-ignored; the deploy job downloads it).